# Resume Classification Project

This notebook demonstrates a multi-model resume classification system that:
1. Extracts structured information from resumes (PDF/DOCX/TXT)
2. Uses spaCy for intelligent feature extraction
3. Classifies resumes into career clusters using both:
   - Bag of Words (BoW) machine learning model
   - Gemini LLM for contextual classification using spaCy feature extraction
4. Compares results from both classification methods

## Architecture
- **Extraction Layer**: spaCy + regex-based section parsing
- **ML Model**: Logistic Regression with BoW features
- **LLM Enhancement**: Google Gemini for nuanced classification
"""

In [ ]:
# Install dependencies
!pip install pandas numpy scikit-learn
!pip install spacy pdfplumber python-docx PyPDF2
!pip install google-generativeai
!python -m spacy download en_core_web_sm

## Step 1: Train Logistic Regression model



In [82]:
import subprocess
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Train and evaluate classification model
print("=" * 60)
print("STEP 1: Training and evaluating classification model...")
print("=" * 60)
result = subprocess.run(
    ["python", "../models/bag_of_words/bag_of_words_model.py"],
    cwd=Path.cwd(),
    capture_output=True,
    text=True
)
print(result.stdout)
# Only print stderr if there's an actual error (non-zero exit code)
if result.returncode != 0 and result.stderr:
    print("Errors:", result.stderr)

STEP 1: Training and evaluating classification model...
Training Bag of Words model...

Evaluating model...
Accuracy: 0.9189

Classification Report:
                          precision    recall  f1-score   support

     Backend Development       0.88      0.86      0.87       191
 Content & Documentation       0.92      0.92      0.92        24
    Data Science & AI/ML       0.92      0.94      0.93       155
 Database Administration       0.84      0.88      0.86        48
          DevOps & Cloud       1.00      0.97      0.99        75
  Engineering Leadership       1.00      1.00      1.00        16
    Frontend Development       0.82      0.84      0.83        95
  Full Stack Development       1.00      1.00      1.00        54
      Mobile Development       1.00      1.00      1.00         9
       Quality Assurance       0.92      0.97      0.94        61
                Security       0.98      1.00      0.99        42
Specialized Technologies       1.00      0.92      0.96   

## Step 2: Extract & Process Resumes

Run the file reader to extract text from PDF/DOCX files and create processed .txt files.


In [83]:
import subprocess
from pathlib import Path

# Run file reader to extract and process resumes
print("=" * 60)
print("STEP 1: Extracting and processing resumes...")
print("=" * 60)
result = subprocess.run(
    ["python", "../models/file_reading_application/file_reader.py"],
    cwd=Path.cwd(),
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)


STEP 1: Extracting and processing resumes...
Starting file processing...
Processed: aws.docx
Processed: Bickford-Resume.pdf
Processed: Brown-Resume.pdf
Processed: Brown-v2-Resume.pdf
Processed: database-admin.docx
Processed: Fantigrossi-CV.pdf
Processed: network-engineer.docx
Processed: old-example.docx
Processed: old-example2.docx
Processed: Thiele-Resume.pdf
Processed: Tintera-Resume.pdf

Starting file processing...
Processed: aws.docx
Processed: Bickford-Resume.pdf
Processed: Brown-Resume.pdf
Processed: Brown-v2-Resume.pdf
Processed: database-admin.docx
Processed: Fantigrossi-CV.pdf
Processed: network-engineer.docx
Processed: old-example.docx
Processed: old-example2.docx
Processed: Thiele-Resume.pdf
Processed: Tintera-Resume.pdf



## Step 2: Run BOW Model Classification

Classify processed resumes using the Bag of Words Logistic Regression model.


In [84]:
print("=" * 60)
print("STEP 2: Running BOW model classification...")
print("=" * 60)
result = subprocess.run(
    ["python", "script/test_model.py"],
    cwd=Path.cwd(),
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)


STEP 2: Running BOW model classification...

Errors: python: can't open file 'd:\\Git\\Resume-Parser\\notebooks\\script\\test_model.py': [Errno 2] No such file or directory



## Step 3: Compare BOW vs spaCy/Gemini

Generate comparison metrics between BOW model predictions and spaCy+Gemini classifications.


In [85]:
print("=" * 60)
print("STEP 3: Comparing BOW and spaCy/Gemini results...")
print("=" * 60)
result = subprocess.run(
    ["python", "script/compare_bow_spacy_gemini.py"],
    cwd=Path.cwd(),
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)


STEP 3: Comparing BOW and spaCy/Gemini results...

Errors: python: can't open file 'd:\\Git\\Resume-Parser\\notebooks\\script\\compare_bow_spacy_gemini.py': [Errno 2] No such file or directory



## Step 4: Explore Results

Load and visualize the outputs from each step.


In [86]:
# Load BOW results
bow_results_path = Path('script/bow_results.json')
if bow_results_path.exists():
    with open(bow_results_path) as f:
        bow_results = json.load(f)
    print(f"\n✓ Loaded {len(bow_results)} BOW predictions from {bow_results_path}")
    
    # Display first few results
    bow_df = pd.DataFrame([
        {
            'file_name': r['file_name'],
            'prediction': r['bow_prediction'],
            'confidence': f"{r['bow_confidence']:.2%}",
            'top_3': ' → '.join([cat['category'] for cat in r['top_3_categories']])
        }
        for r in bow_results[:5]
    ])
    print("\nFirst 5 BOW predictions:")
    print(bow_df.to_string(index=False))

# Load comparison results
comp_path = Path('script/comparison_details.json')
if comp_path.exists():
    with open(comp_path) as f:
        comparison = json.load(f)
    print(f"\n✓ Loaded comparison data: {len(comparison)} files analyzed")
    
    # Summary stats
    top1_matches = sum(1 for c in comparison if c['top1_match'])
    total_overlap = sum(c['top3_overlap'] for c in comparison)
    print(f"\nComparison Summary:")
    print(f"  - Top-1 agreement: {top1_matches}/{len(comparison)} ({top1_matches/len(comparison):.1%})")
    print(f"  - Average top-3 overlap: {total_overlap/len(comparison):.2f} items")
